In [1]:
import json
from pydantic import BaseModel, Field
from typing import List, Dict, Type, Optional
from openai import OpenAI

In [2]:
CLIENT = OpenAI(base_url="http://127.0.0.1:8001/v1", api_key="none")

In [3]:
with open("prompts/taxonomy_lvl_1_prompt.txt", "r", encoding="utf-8") as f:
    TYPE_SEGREGATION_PROMPT = f.read()

In [11]:
llm_inputs = []
for i in range(1):
    with open(f"entity_output_files/chapter_{i}_output.json", "r") as f:
        all_resolved_sentences = []
        all_extracted_entities = []
        data = json.load(f)
        print(f"chapter {i} - {len(data)} groups")
        for group in data:
            # print(json.dumps(group['extracted_entities'], indent=4, ensure_ascii=False))
            resolved_sentences = [obj['sentence'] for obj in group['extracted_entities']]
            extracted_entities = [entity for obj in group['extracted_entities'] for entity in obj['entities']]
            extracted_entities = list(set(extracted_entities))
            # all_resolved_sentences.extend(resolved_sentences)
            # all_extracted_entities.extend(extracted_entities)
            # resolved_sentences = [sentence for obj in group['extracted_entities'] for sentence in obj['sentence']]
            llm_inputs.append({
                "resolved_sentences": resolved_sentences,
                "extracted_entities": extracted_entities
            })

        # llm_inputs.append({
        #     "resolved_sentences": resolved_sentences,
        #     "extracted_entities": extracted_entities
        # })
        
        # print("Extracted Entities:\n", json.dumps(extracted_entities, indent=4, ensure_ascii=False))
        # data["extracted_entities"] = extracted_entities
        # llm_inputs.append(data)

chapter 0 - 5 groups


In [12]:
len(llm_inputs)

5

In [13]:
class Category_lvl_1(BaseModel):
    categories: Dict[str, List[str]] = Field(
        description="A dictionary mapping dynamically generated broad categories to arrays of entities. The keys of the dictionary are the broad categories, and the values are lists of entities that belong to each category."
    )

In [7]:
def parse_thought_and_response(full_text):
    """
    Helper function to separate the model's internal reasoning from the final answer.
    """
    if "<|channel>thought" in full_text and "<channel|>" in full_text:
        # Split the text around the closing tag
        parts = full_text.split("<channel|>")
        
        # Clean up the thought process (removing the opening tag)
        thought_process = parts[0].replace("<|channel>thought", "").strip()
        
        # The final answer is everything after the closing tag
        final_answer = parts[1].strip()
        
        return {
            "thought_process": thought_process,
            "final_answer": final_answer,
            "raw_output": full_text
        }
    
    # Fallback if the model didn't output thought tags for some reason
    return {
        "thought_process": None,
        "final_answer": full_text.strip(),
        "raw_output": full_text
    }

In [8]:
def generate_with_thinking(
    system_prompt: str, 
    user_query: str, 
    response_model: Optional[Type[BaseModel]] = None, # Defaults to None
    enable_thinking: bool = False
):
    # Enable thinking by injecting the token at the start of the system instructions
    system_prompt = f"<|think|>\n{system_prompt}" if enable_thinking else system_prompt

    # 1. Define the base parameters that are always required
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192, 
        "temperature": 0.2,
    }

    # 2. Conditionally inject the response_format only if a model is provided
    if response_model is not None:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema()
            }
        }

    # 3. Unpack the dictionary into the API call using **
    response = CLIENT.chat.completions.create(**api_params)
    
    full_response = response.choices[0].message.content
    return parse_thought_and_response(full_response)

In [14]:
llm_outputs = []
result = None

In [15]:
len(llm_outputs)

0

In [ ]:
# llm_inputs[1]

In [ ]:
# print(result['final_answer'])

In [16]:
import time

max_retries = 3
failed_queries = []

for i in range(len(llm_inputs)):
    user_query = json.dumps(llm_inputs[i], ensure_ascii=False, indent=4)
    
    for attempt in range(max_retries):
        try:
            result = generate_with_thinking(
                TYPE_SEGREGATION_PROMPT + "\n\ninput: \n\n", 
                user_query, 
                response_model=Category_lvl_1
            )
            parsed_result = Category_lvl_1.model_validate_json(result["final_answer"])
            llm_outputs.append(parsed_result.model_dump())
            
            # If successful, break out of the retry loop and move to the next input
            break
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for input index {i}: {e}")
            
            # If this was the last attempt, log the failure and let the loop move on
            if attempt == max_retries - 1:
                print(f"Input {i} failed after {max_retries} attempts. Skipping.")
                failed_queries.append({
                    "index": i,
                    "query": user_query,
                    "error": str(e)
                })
            else:
                # Optional: Add a brief pause before retrying to let the API/server recover
                time.sleep(2) 

# You can check failed_queries after the loop finishes
if failed_queries:
    print(f"\nTotal failed queries: {len(failed_queries)}")

Attempt 1 failed for input index 0: 1 validation error for Category_lvl_1
  Invalid JSON: EOF while parsing a string at line 18 column 23499 [type=json_invalid, input_value='{\n  "categories": {\n  ...काल", "प्र', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
Attempt 2 failed for input index 0: 1 validation error for Category_lvl_1
  Invalid JSON: EOF while parsing a string at line 5 column 22643 [type=json_invalid, input_value='{\n  "categories": {\n  ...वयव", "काल', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
Attempt 1 failed for input index 1: 1 validation error for Category_lvl_1
  Invalid JSON: EOF while parsing a string at line 12 column 20347 [type=json_invalid, input_value='{\n  "categories": {\n  ...ीय त्योह', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
Attempt 1 failed for input index 2: 1 validation error for Category

In [ ]:
len(llm_inputs), len(llm_outputs), len(failed_queries)

In [ ]:
json_output = json.dumps(llm_outputs, ensure_ascii=False, indent=4)
with open("raw_categories_output.json", "w", encoding="utf-8") as f:
    f.write(json_output)

In [ ]:
raw_category_dict = {}
for chapter in llm_outputs:
    for key in chapter['categories'].keys():
        if key not in raw_category_dict:
            raw_category_dict[key] = 0
        else:
            raw_category_dict[key] += 1
        

In [ ]:
raw_keys = list(raw_category_dict.keys())

In [ ]:
len(raw_keys)

In [ ]:
with open("raw_categories_output.json", "r", encoding="utf-8") as f:
    raw_categories = json.load(f)

In [ ]:
raw_category_flattened = []
for chapter in raw_categories:
    raw_category_flattened.append(chapter['categories'])

with open("raw_category_flattened.json", "w", encoding="utf-8") as f:
    json.dump(raw_category_flattened, f, ensure_ascii=False, indent=4)

In [11]:
class CategoryClustering(BaseModel):
    clustered_categories: List[List[str]] = Field(
        description="A list of lists. Each inner list contains semantically merged category names. Standalone categories are in a list by themselves."
    )

In [12]:
with open("prompts/standardization_lvl_1_prompt.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# 1. Package your variables into the exact JSON structure the prompt expects
input_payload = {
    "target_categories_to_group": raw_keys,
    "category_frequencies": raw_category_dict,
    "contextual_data": raw_category_flattened
}

# 2. Convert to a formatted string (ensure_ascii=False is crucial for Hindi)
user_query = json.dumps(input_payload, ensure_ascii=False, indent=4)

print("Sending categorization task to LLM...")

# 3. Call your refactored function
result = generate_with_thinking(
    system_prompt=SYSTEM_PROMPT,
    user_query=user_query,
    response_model=CategoryClustering, # Pass the Pydantic class
    enable_thinking=True               # Give the model time to reason about semantics
)

# 4. Parse the validated JSON string into the Pydantic object
parsed_taxonomy = CategoryClustering.model_validate_json(result["final_answer"])

# 5. Extract your final list of lists
final_clusters = parsed_taxonomy.clustered_categories

# --- Optional but highly recommended: Sanity Check ---
# Verify that no categories were lost or hallucinated during the LLM's generation
original_set = set(raw_keys)
output_set = set([item for sublist in final_clusters for item in sublist])

if original_set == output_set:
    print(f"Success! Grouped {len(raw_keys)} raw categories into {len(final_clusters)} semantic clusters.")
else:
    missing = original_set - output_set
    hallucinated = output_set - original_set
    if missing:
        print(f"Warning: LLM dropped these categories: {missing}")
    if hallucinated:
        print(f"Warning: LLM invented these categories: {hallucinated}")

# View the final output
for i, cluster in enumerate(final_clusters):
    print(f"Group {i+1}: {cluster}")

NameError: name 'raw_keys' is not defined

In [13]:
with open("prompts/recategorization_lvl_1.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

In [14]:
class ExtractedEntities(BaseModel):
    # We use Optional[List[str]] = None so empty categories can just be omitted
    Mythical_Entity: Optional[List[str]] = Field(default=None)
    Celestial_Entity: Optional[List[str]] = Field(default=None)
    Phenomenon: Optional[List[str]] = Field(default=None)
    Time: Optional[List[str]] = Field(default=None)
    Food: Optional[List[str]] = Field(default=None)
    Activity: Optional[List[str]] = Field(default=None)
    Concept: Optional[List[str]] = Field(default=None)
    Object: Optional[List[str]] = Field(default=None)
    Living_Being: Optional[List[str]] = Field(default=None)
    Text: Optional[List[str]] = Field(default=None)
    Location: Optional[List[str]] = Field(default=None)
    Primordial_Element: Optional[List[str]] = Field(default=None)
    Medical_Concept: Optional[List[str]] = Field(default=None)
    Geographical_Feature: Optional[List[str]] = Field(default=None)
    Event: Optional[List[str]] = Field(default=None)
    Emotions: Optional[List[str]] = Field(default=None)
    
    # Use alias to handle the ampersand in the JSON key!
    Social_Group_and_Role: Optional[List[str]] = Field(
        default=None, 
        alias="Social_Group_&_Role"
    )
    
    Body_part: Optional[List[str]] = Field(default=None)
    Sanskrit_text: Optional[List[str]] = Field(default=None)
    Other: Optional[List[str]] = Field(default=None)

In [15]:
categories_lvl_1 = []

In [16]:
result = None

In [17]:
len(categories_lvl_1)

0

In [ ]:
import time
import json
from collections import defaultdict

# 1. Initialize data structures
global_entity_registry = defaultdict(set)
global_expected_entities = set() # To track all valid entities fed to the LLM

failed_sentences = []
validation_flags = [] # To store missing/extra entity warnings
max_retries = 3

print("Starting sentence-level entity categorization with validation...")

for i in range(25): # Assuming 25 chapters
    try:
        with open(f"entity_output_files/chapter_{i}_output.json", "r", encoding='utf-8') as f:
            data = json.load(f)
            
            for group_idx, group in enumerate(data):
                
                # Iterate through each individual sentence object within the group
                for sent_idx, sentence_obj in enumerate(group.get('extracted_entities', [])):
                    
                    # The payload is now just: {"sentence": "...", "entities": ["..."]}
                    base_user_query = json.dumps(sentence_obj, ensure_ascii=False, indent=4)
                    success = False
                    
                    # Extract the EXPECTED entities for THIS specific sentence
                    expected_entities = set(sentence_obj.get('entities', []))
                    
                    # Skip if there are no entities in this sentence to save API calls
                    if not expected_entities:
                        continue
                        
                    prompt_suffix = "" 
                    
                    for attempt in range(max_retries):
                        try:
                            # Append any feedback from previous failed attempts to the query
                            current_query = base_user_query + prompt_suffix
                            
                            result = generate_with_thinking(
                                SYSTEM_PROMPT + "\n\ninput: \n\n", 
                                current_query, 
                                response_model=ExtractedEntities,
                                enable_thinking=True
                            )
                            
                            # Validate and parse the JSON
                            parsed_result = ExtractedEntities.model_validate_json(result["final_answer"])
                            category_dict = parsed_result.model_dump()
                            
                            # Gather the RETURNED entities for validation
                            returned_entities = set()
                            for entity_list in category_dict.values():
                                if entity_list:
                                    returned_entities.update(entity_list)
                            
                            # --- VALIDATION 1: SENTENCE LEVEL MISSING CHECK ---
                            missing_entities = expected_entities - returned_entities
                            
                            # If entities are missing AND we still have retries left:
                            if missing_entities and attempt < max_retries - 1:
                                missing_list_str = json.dumps(list(missing_entities), ensure_ascii=False)
                                # Update the suffix to scold the LLM on the next attempt
                                prompt_suffix = f"\n\nCRITICAL FEEDBACK FROM LAST ATTEMPT:\nYou forgot to categorize these exact entities: {missing_list_str}.\nYou MUST include them in your final JSON output."
                                
                                # Throw a custom error to trigger the except block and retry
                                raise ValueError(f"Missing {len(missing_entities)} entities. Forcing retry.")
                            
                            
                            # --- SUCCESS OR FINAL ATTEMPT ACCEPTANCE ---
                            # 2. Merge this sentence's entities into the Global Registry
                            for category_name, entity_list in category_dict.items():
                                if entity_list: 
                                    global_entity_registry[category_name].update(entity_list)
                            
                            # If it is the last attempt and STILL missing, log the flag
                            if missing_entities:
                                warning_msg = f"Chapter {i}, Group {group_idx}, Sent #{sent_idx} missing {len(missing_entities)} entities: {missing_entities}"
                                print(f"  [FLAG - MISSING] {warning_msg}")
                                validation_flags.append({
                                    "type": "missing_in_sentence",
                                    "chapter": i,
                                    "group": group_idx,
                                    "sentence_idx": sent_idx,
                                    "sentence": sentence_obj.get("sentence", ""),
                                    "entities": list(missing_entities)
                                })
                            
                            # Add to global tracker for final validation
                            global_expected_entities.update(expected_entities)
                            
                            success = True
                            break # Break the retry loop on success
                            
                        except Exception as e:
                            print(f"Chapter {i}, Group {group_idx}, Sent {sent_idx} - Attempt {attempt + 1} failed: {e}")
                            if attempt < max_retries - 1:
                                time.sleep(1) # Give vLLM a brief moment to breathe
                                
                    if not success:
                        print(f"Skipping Chapter {i}, Group {group_idx}, Sent {sent_idx} after {max_retries} completely failed attempts.")
                        failed_sentences.append({
                            "chapter": i, 
                            "group": group_idx, 
                            "sentence_idx": sent_idx,
                            "query": base_user_query
                        })

    except FileNotFoundError:
        print(f"File for chapter {i} not found. Skipping.")


# 3. Convert the sets back into standard lists so it can be exported as JSON
final_global_dictionary = {
    category: list(entities) 
    for category, entities in global_entity_registry.items()
}

# --- VALIDATION 2: GLOBAL EXTRA/HALLUCINATION CHECK ---
global_returned_entities = set()
for entities in global_entity_registry.values():
    global_returned_entities.update(entities)

extra_entities = global_returned_entities - global_expected_entities

if extra_entities:
    print(f"\n[FLAG - HALLUCINATION] The LLM generated {len(extra_entities)} extra entities not present in the source.")
    validation_flags.append({
        "type": "global_extra_entities",
        "entities": list(extra_entities)
    })
else:
    print("\n[SUCCESS] No hallucinated entities found in final categorization.")

global_missing = global_expected_entities - global_returned_entities
if not global_missing:
    print("[SUCCESS] All source entities were successfully preserved in the Knowledge Graph.")

print(f"\nProcessing complete. Failed sentences: {len(failed_sentences)} | Validation Flags Raised: {len(validation_flags)}")

# Save the final Knowledge Graph
with open("final_global_entities.json", "w", encoding="utf-8") as f:
    json.dump(final_global_dictionary, f, ensure_ascii=False, indent=4)

# Save the Validation Logs for review
with open("validation_error_logs.json", "w", encoding="utf-8") as f:
    log_data = {
        "failed_api_calls": failed_sentences,
        "validation_flags": validation_flags
    }
    json.dump(log_data, f, ensure_ascii=False, indent=4)

Starting sentence-level entity categorization with validation...
Chapter 0, Group 0, Sent 11 - Attempt 1 failed: 1 validation error for ExtractedEntities
  Invalid JSON: EOF while parsing a string at line 7 column 21349 [type=json_invalid, input_value='{\n  "Text": ["धर्...यव", "पूजन', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
Chapter 0, Group 0, Sent 28 - Attempt 1 failed: Missing 2 entities. Forcing retry.
Chapter 0, Group 0, Sent 41 - Attempt 1 failed: Missing 1 entities. Forcing retry.
Chapter 0, Group 0, Sent 47 - Attempt 1 failed: Missing 1 entities. Forcing retry.
Chapter 0, Group 1, Sent 5 - Attempt 1 failed: Missing 2 entities. Forcing retry.
Chapter 0, Group 1, Sent 10 - Attempt 1 failed: Missing 1 entities. Forcing retry.
Chapter 0, Group 1, Sent 14 - Attempt 1 failed: Missing 1 entities. Forcing retry.
Chapter 0, Group 1, Sent 14 - Attempt 2 failed: Missing 1 entities. Forcing retry.
  [FLAG - MISSING] Chapter 0, Group

In [ ]:
# parsed_result = ExtractedEntities.model_validate_json(result["final_answer"])
# category_dict = parsed_result.model_dump()
# print(json.dumps(category_dict, indent=4, ensure_ascii=False))
# for k,v in category_dict:
#     category_dict[k] = list(set(v))

In [ ]:
# print(result['final_answer'])

In [18]:
print(json.dumps(categories_lvl_1, ensure_ascii=False, indent=4))

[
    {
        "Mythical_Entity": [
            "ब्रह्माजी",
            "मार्तण्ड",
            "ब्रह्मा"
        ],
        "Celestial_Entity": [
            "सूर्य",
            "राशि",
            "मेष",
            "राशि-चक्र"
        ],
        "Phenomenon": [
            "सूर्योदय",
            "सृष्टि",
            "उत्पत्ति",
            "ऋतुपरिवर्तन"
        ],
        "Time": [
            "मार्गशीर्ष मास",
            "चैत्रमास",
            "संवत्सरारम्भ",
            "शुक्ल पक्ष",
            "वसन्त",
            "चैत्र शुक्ल प्रतिपदा",
            "शरद्",
            "तिथि",
            "आग्रहायणिक",
            "शिशिर",
            "ग्रीष्म",
            "वर्षा ऋतु",
            "वैदिक काल",
            "संवत्सर",
            "अधिक मास",
            "हेमन्त",
            "वर्ष",
            "वसंत ऋतु",
            "काल-विज्ञान",
            "समय",
            "दिन",
            "निरयन पक्ष",
            "वर्षा",
            "ऋतु",
            "सायन पक्ष",
            "